# NeuroScan AI — Federated Brain Tumor Detection (Google Colab)

Complete research-development notebook using **3 simulated hospitals**, **VGG16**, **FedAvg**, **Grad-CAM**, and full evaluation.

Datasets:
- Hospital 1: `masoudnickparvar/brain-tumor-mri-dataset`
- Hospital 2: `ahmedhamada0/brain-tumor-detection` (Br35H)
- Hospital 3: `navoneel/brain-mri-images-for-brain-tumor-detection`

Task: **No Tumor vs Tumor**.

> Academic/research prototype only; not for clinical diagnosis.

## Step 1 — Check GPU

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## Step 2 — Install packages

In [ ]:
!pip install -q kagglehub scikit-learn opencv-python-headless matplotlib pandas

## Step 3 — Imports and seed

In [ ]:
import os, gc, json, random, shutil, hashlib
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from tensorflow.keras import Model, layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import load_img, img_to_array

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("Ready")

## Step 4 — Download all 3 datasets

In [ ]:
import kagglehub
hospital1_path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
hospital2_path = kagglehub.dataset_download("ahmedhamada0/brain-tumor-detection")
hospital3_path = kagglehub.dataset_download("navoneel/brain-mri-images-for-brain-tumor-detection")
print("H1:", hospital1_path)
print("H2:", hospital2_path)
print("H3:", hospital3_path)

## Step 5 — Inspect downloaded folders

In [ ]:
def show_structure(root, max_items=60):
    root=Path(root); shown=0
    print("\nROOT:", root)
    for p in root.rglob("*"):
        if shown>=max_items: break
        rel=p.relative_to(root)
        if len(rel.parts)<=4:
            print(rel); shown+=1

show_structure(hospital1_path)
show_structure(hospital2_path)
show_structure(hospital3_path)

## Step 6 — Image helpers and label detection

In [ ]:
IMAGE_EXTENSIONS={".jpg",".jpeg",".png",".bmp",".gif"}
def is_image(p): return Path(p).suffix.lower() in IMAGE_EXTENSIONS
def get_all_images(root): return [p for p in Path(root).rglob("*") if p.is_file() and is_image(p)]
def count_images(folder):
    folder=Path(folder)
    return sum(1 for p in folder.rglob("*") if p.is_file() and is_image(p)) if folder.exists() else 0

def norm(s): return str(s).lower().replace(" ","").replace("_","").replace("-","")
NEG={"no","notumor","notumour","nontumor","nontumour","normal","healthy","negative"}
POS={"yes","tumor","tumour","positive","glioma","gliomatumor","gliomatumour","meningioma","meningiomatumor","meningiomatumour","pituitary","pituitarytumor","pituitarytumour"}

def detect_label(path):
    parts=[norm(x) for x in Path(path).parent.parts]
    for x in parts:
        if x in NEG: return "no_tumor"
    for x in parts:
        if x in POS: return "tumor"
    return None

for name,path in [("H1",hospital1_path),("H2",hospital2_path),("H3",hospital3_path)]:
    print(name, "candidate images:", len(get_all_images(path)))

## Step 7 — Build standardized Hospital 1/2/3 folders

In [ ]:
FED_ROOT=Path("/content/federated_brain_data")
if FED_ROOT.exists(): shutil.rmtree(FED_ROOT)
hospitals=["hospital_1","hospital_2","hospital_3"]
classes=["no_tumor","tumor"]
for h in hospitals:
    for c in classes:
        (FED_ROOT/h/c).mkdir(parents=True,exist_ok=True)

def copy_dataset(source,hospital,prefix,skip_pred=False):
    counts={"tumor":0,"no_tumor":0,"unknown":0,"pred_skipped":0}
    for p in get_all_images(source):
        parts=[norm(x) for x in p.parts]
        if skip_pred and any(x in {"pred","prediction","predictions"} for x in parts):
            counts["pred_skipped"]+=1; continue
        label=detect_label(p)
        if label is None:
            counts["unknown"]+=1; continue
        out=FED_ROOT/hospital/label/f"{prefix}_{counts[label]:07d}{p.suffix.lower()}"
        shutil.copy2(p,out); counts[label]+=1
    print(hospital, counts)
    return counts

copy_dataset(hospital1_path,"hospital_1","H1",False)
copy_dataset(hospital2_path,"hospital_2","H2",True)
copy_dataset(hospital3_path,"hospital_3","H3",False)

## Step 8 — Verify both classes exist

In [ ]:
for h in hospitals:
    a=count_images(FED_ROOT/h/"no_tumor"); b=count_images(FED_ROOT/h/"tumor")
    print(h, "no_tumor=",a,"tumor=",b,"total=",a+b)
    assert a>0 and b>0, f"Empty class detected in {h}"
print("Verification passed")

## Step 9 — Remove exact cross-hospital duplicates (recommended)

In [ ]:
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

seen={}; removed=0
for hospital in hospitals:
    for label in classes:
        for p in sorted((FED_ROOT/hospital/label).iterdir()):
            if not p.is_file() or not is_image(p): continue
            d=sha256_file(p)
            if d in seen:
                p.unlink(); removed+=1
            else:
                seen[d]=str(p)
print("Exact duplicates removed:", removed)

## Step 10 — 80/10/10 split per hospital

In [ ]:
SPLIT_ROOT=Path("/content/federated_split")
if SPLIT_ROOT.exists(): shutil.rmtree(SPLIT_ROOT)
for h in hospitals:
    for s in ["train","val","test"]:
        for c in classes:
            (SPLIT_ROOT/h/s/c).mkdir(parents=True,exist_ok=True)

def split_hospital(h):
    rng=random.Random(SEED)
    for label in classes:
        images=[p for p in (FED_ROOT/h/label).iterdir() if p.is_file() and is_image(p)]
        rng.shuffle(images); n=len(images)
        if n<10: raise ValueError(f"Not enough images in {h}/{label}: {n}")
        n_train=int(.8*n); n_val=int(.1*n)
        groups={"train":images[:n_train],"val":images[n_train:n_train+n_val],"test":images[n_train+n_val:]}
        for split,files_ in groups.items():
            for p in files_: shutil.copy2(p,SPLIT_ROOT/h/split/label/p.name)
        print(h,label,{k:len(v) for k,v in groups.items()})
for h in hospitals: split_hospital(h)

## Step 11 — Verify split before TensorFlow

In [ ]:
for h in hospitals:
    print("\n",h)
    for s in ["train","val","test"]:
        n0=count_images(SPLIT_ROOT/h/s/"no_tumor"); n1=count_images(SPLIT_ROOT/h/s/"tumor")
        print(s,"no_tumor=",n0,"tumor=",n1,"total=",n0+n1)
        assert n0>0 and n1>0
print("All split folders are valid")

## Step 12 — TensorFlow data pipeline

In [ ]:
IMG_SIZE=(224,224); BATCH_SIZE=16
augmentation=tf.keras.Sequential([layers.RandomFlip("horizontal"),layers.RandomRotation(0.05),layers.RandomZoom(0.10)])

def create_dataset(directory,training=True):
    directory=Path(directory); n=count_images(directory)
    print("Loading",directory,"images=",n)
    if n==0: raise ValueError(f"No images in {directory}")
    ds=tf.keras.utils.image_dataset_from_directory(directory,labels="inferred",label_mode="binary",class_names=["no_tumor","tumor"],image_size=IMG_SIZE,batch_size=BATCH_SIZE,shuffle=training,seed=SEED)
    if training:
        ds=ds.map(lambda x,y:(preprocess_input(augmentation(tf.cast(x,tf.float32),training=True)),y),num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds=ds.map(lambda x,y:(preprocess_input(tf.cast(x,tf.float32)),y),num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_datasets={}; val_datasets={}; test_datasets={}
for h in hospitals:
    train_datasets[h]=create_dataset(SPLIT_ROOT/h/"train",True)
    val_datasets[h]=create_dataset(SPLIT_ROOT/h/"val",False)
    test_datasets[h]=create_dataset(SPLIT_ROOT/h/"test",False)
print("TensorFlow datasets created successfully")

## Step 13 — VGG16 global/local model

In [ ]:
FEDERATED_ROUNDS=5; LOCAL_EPOCHS=1; LEARNING_RATE=1e-4

def build_model(imagenet=True):
    base=VGG16(weights="imagenet" if imagenet else None,include_top=False,input_shape=(224,224,3),name="vgg16")
    base.trainable=False
    x=layers.GlobalAveragePooling2D(name="global_average_pool")(base.output)
    x=layers.Dense(256,activation="relu",name="fc1")(x)
    x=layers.Dense(128,activation="relu",name="fc2")(x)
    out=layers.Dense(1,activation="sigmoid",name="output")(x)
    m=Model(base.input,out,name="NeuroScan_VGG16")
    m.compile(optimizer=Adam(LEARNING_RATE),loss="binary_crossentropy",metrics=["accuracy",tf.keras.metrics.Precision(name="precision"),tf.keras.metrics.Recall(name="recall")])
    return m

global_model=build_model(True)
global_model.summary()

## Step 14 — FedAvg helpers

In [ ]:
client_sizes={h:count_images(SPLIT_ROOT/h/"train") for h in hospitals}
print(client_sizes)

def fedavg(client_weights,client_sizes_):
    total=float(sum(client_sizes_)); out=[]
    for layer_group in zip(*client_weights):
        layer=np.zeros_like(layer_group[0],dtype=np.float32)
        for w,n in zip(layer_group,client_sizes_): layer += w.astype(np.float32)*(n/total)
        out.append(layer)
    return out

global_val_dataset=val_datasets["hospital_1"].concatenate(val_datasets["hospital_2"]).concatenate(val_datasets["hospital_3"])

## Step 15 — Federated training

In [ ]:
history={"round":[],"hospital_1_accuracy":[],"hospital_2_accuracy":[],"hospital_3_accuracy":[],"global_validation_accuracy":[]}
global_weights=global_model.get_weights()

for r in range(1,FEDERATED_ROUNDS+1):
    print("\n"+"="*70); print(f"FEDERATED ROUND {r}/{FEDERATED_ROUNDS}"); print("="*70)
    weight_sets=[]; size_list=[]; accs={}
    for h in hospitals:
        print("Training",h)
        local=build_model(False); local.set_weights(global_weights)
        hist=local.fit(train_datasets[h],validation_data=val_datasets[h],epochs=LOCAL_EPOCHS,verbose=1)
        accs[h]=float(hist.history["accuracy"][-1])
        weight_sets.append(local.get_weights()); size_list.append(client_sizes[h])
        del local; gc.collect()
    global_weights=fedavg(weight_sets,size_list); global_model.set_weights(global_weights)
    val_result=global_model.evaluate(global_val_dataset,verbose=0); gacc=float(val_result[1])
    history["round"].append(r)
    for h in hospitals: history[f"{h}_accuracy"].append(accs[h])
    history["global_validation_accuracy"].append(gacc)
    print("Global validation accuracy:",round(gacc*100,2),"%")

history_df=pd.DataFrame(history); display(history_df)

## Step 16 — Plot federated accuracy and save model

In [ ]:
plt.figure(figsize=(10,6))
for h in hospitals:
    plt.plot(history["round"],np.array(history[f"{h}_accuracy"])*100,marker="o",label=h)
plt.plot(history["round"],np.array(history["global_validation_accuracy"])*100,marker="o",linewidth=3,label="global")
plt.xlabel("Federated Round"); plt.ylabel("Accuracy (%)"); plt.title("Federated Learning Accuracy"); plt.legend(); plt.grid(); plt.show()

MODEL_DIR=Path("/content/models"); MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH=MODEL_DIR/"neuroscan_fedavg_vgg16.keras"
global_model.save(MODEL_PATH)
print("Saved:",MODEL_PATH)

## Step 17 — Global test evaluation

In [ ]:
global_test_dataset=test_datasets["hospital_1"].concatenate(test_datasets["hospital_2"]).concatenate(test_datasets["hospital_3"])
y_true=[]; y_prob=[]
for images,labels in global_test_dataset:
    p=global_model.predict(images,verbose=0).flatten()
    y_prob.extend(p); y_true.extend(labels.numpy().flatten().astype(int))
y_true=np.array(y_true); y_prob=np.array(y_prob); y_pred=(y_prob>=0.5).astype(int)
accuracy=accuracy_score(y_true,y_pred); precision=precision_score(y_true,y_pred,zero_division=0); recall=recall_score(y_true,y_pred,zero_division=0); f1=f1_score(y_true,y_pred,zero_division=0)
auc=roc_auc_score(y_true,y_prob) if len(np.unique(y_true))==2 else float("nan")
print(f"Accuracy : {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall   : {recall*100:.2f}%")
print(f"F1 Score : {f1*100:.2f}%")
print(f"ROC-AUC  : {auc:.4f}")
print(classification_report(y_true,y_pred,target_names=["No Tumor","Tumor"],zero_division=0))

## Step 18 — Confusion matrix and ROC curve

In [ ]:
cm=confusion_matrix(y_true,y_pred)
plt.figure(figsize=(6,5)); plt.imshow(cm); plt.title("Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.xticks([0,1],["No Tumor","Tumor"]); plt.yticks([0,1],["No Tumor","Tumor"])
for i in range(2):
    for j in range(2): plt.text(j,i,cm[i,j],ha="center",va="center",fontsize=15)
plt.colorbar(); plt.show()

if len(np.unique(y_true))==2:
    fpr,tpr,_=roc_curve(y_true,y_prob)
    plt.figure(figsize=(7,6)); plt.plot(fpr,tpr,label=f"AUC={auc:.4f}"); plt.plot([0,1],[0,1],linestyle="--"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC Curve"); plt.legend(); plt.grid(); plt.show()

## Step 19 — Per-hospital test results

In [ ]:
hospital_results={}
for h in hospitals:
    yt=[]; yp=[]
    for images,labels in test_datasets[h]:
        p=global_model.predict(images,verbose=0).flatten(); yp.extend(p); yt.extend(labels.numpy().flatten().astype(int))
    yt=np.array(yt); yp=np.array(yp); pred=(yp>=0.5).astype(int)
    hospital_results[h]={"accuracy":float(accuracy_score(yt,pred)),"precision":float(precision_score(yt,pred,zero_division=0)),"recall":float(recall_score(yt,pred,zero_division=0)),"f1":float(f1_score(yt,pred,zero_division=0)),"auc":float(roc_auc_score(yt,yp)) if len(np.unique(yt))==2 else float("nan"),"test_samples":int(len(yt))}
hospital_results_df=pd.DataFrame(hospital_results).T
display(hospital_results_df)

## Step 20 — Grad-CAM

In [ ]:
def make_gradcam(image_batch,model,conv_name="block5_conv3"):
    vgg=model.get_layer("vgg16"); conv=vgg.get_layer(conv_name)
    conv_model=Model(vgg.input,conv.output)
    inp=tf.keras.Input(shape=conv.output.shape[1:]); x=inp; start=False
    for layer in vgg.layers:
        if layer.name==conv_name: start=True; continue
        if start: x=layer(x)
    x=model.get_layer("global_average_pool")(x); x=model.get_layer("fc1")(x); x=model.get_layer("fc2")(x); out=model.get_layer("output")(x)
    classifier=Model(inp,out)
    with tf.GradientTape() as tape:
        conv_out=conv_model(image_batch); tape.watch(conv_out); score=classifier(conv_out)[:,0]
    grads=tape.gradient(score,conv_out); pooled=tf.reduce_mean(grads,axis=(0,1,2)); heat=tf.reduce_sum(conv_out[0]*pooled,axis=-1); heat=tf.maximum(heat,0); m=tf.reduce_max(heat)
    return tf.where(m>0,heat/m,heat).numpy()

sample_folder=SPLIT_ROOT/"hospital_1"/"test"/"tumor"
sample=[p for p in sample_folder.iterdir() if p.is_file() and is_image(p)][0]
img=load_img(sample,target_size=IMG_SIZE); arr=img_to_array(img); original=arr.copy(); batch=preprocess_input(np.expand_dims(arr,0))
prob=float(global_model.predict(batch,verbose=0)[0][0]); pred_text="Tumor Detected" if prob>=0.5 else "No Tumor Detected"; confidence=prob if prob>=0.5 else 1-prob
heat=make_gradcam(batch,global_model); heat=cv2.resize(heat,(original.shape[1],original.shape[0])); heat_color=cv2.cvtColor(cv2.applyColorMap(np.uint8(np.clip(heat,0,1)*255),cv2.COLORMAP_JET),cv2.COLOR_BGR2RGB); overlay=np.clip(0.6*original+0.4*heat_color,0,255).astype(np.uint8)
print(pred_text, f"Tumor probability={prob*100:.2f}%", f"Confidence={confidence*100:.2f}%")
plt.figure(figsize=(12,5)); plt.subplot(1,2,1); plt.imshow(original.astype(np.uint8)); plt.title("Original MRI"); plt.axis("off"); plt.subplot(1,2,2); plt.imshow(overlay); plt.title("Grad-CAM"); plt.axis("off"); plt.show()

## Step 21 — Save metrics and CSV files

In [ ]:
ARTIFACT_DIR=Path("/content/artifacts"); ARTIFACT_DIR.mkdir(exist_ok=True)
metrics={"accuracy":float(accuracy),"precision":float(precision),"recall":float(recall),"f1":float(f1),"auc":float(auc),"confusion_matrix":cm.tolist(),"hospital_results":hospital_results,"federated_history":history,"configuration":{"model":"VGG16 + custom head","classes":["No Tumor","Tumor"],"federated_algorithm":"Sample-weighted FedAvg","clients":3,"rounds":FEDERATED_ROUNDS,"local_epochs":LOCAL_EPOCHS,"batch_size":BATCH_SIZE,"learning_rate":LEARNING_RATE,"input_size":"224x224x3","optimizer":"Adam","loss":"Binary Crossentropy","explainability":"Grad-CAM","split":"80/10/10 per hospital"}}
METRICS_PATH=ARTIFACT_DIR/"metrics.json"; HISTORY_PATH=ARTIFACT_DIR/"federated_history.csv"; HOSPITAL_RESULTS_PATH=ARTIFACT_DIR/"hospital_results.csv"
with open(METRICS_PATH,"w") as f: json.dump(metrics,f,indent=2)
history_df.to_csv(HISTORY_PATH,index=False); hospital_results_df.to_csv(HOSPITAL_RESULTS_PATH)
print(METRICS_PATH); print(HISTORY_PATH); print(HOSPITAL_RESULTS_PATH)

## Step 22 — Save final outputs to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PROJECT=Path("/content/drive/MyDrive/NeuroScan_AI"); DRIVE_PROJECT.mkdir(parents=True,exist_ok=True)
for p in [MODEL_PATH,METRICS_PATH,HISTORY_PATH,HOSPITAL_RESULTS_PATH]: shutil.copy2(p,DRIVE_PROJECT/p.name)
print("Saved to:",DRIVE_PROJECT)

## Final files

After successful execution:

- `/content/models/neuroscan_fedavg_vgg16.keras`
- `/content/artifacts/metrics.json`
- `/content/artifacts/federated_history.csv`
- `/content/artifacts/hospital_results.csv`

Google Drive copy:

`MyDrive/NeuroScan_AI/`

Use `neuroscan_fedavg_vgg16.keras` later in the Flask/NeuroScan web application.

**Important:** Report actual measured metrics; do not hard-code a target such as 98.8%.